In [ ]:
# Importing the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Importing the proteins data
prm = []
prmd = []

f = open("../../../../../../Data/Proteins/DENV1/PrM/DENV1_Premembrane.txt", "r")
for x in f:   
    if "DSS" in x:
        prmd.append(1)
    elif "DHF" in x:
        prmd.append(1)
    elif x[0] == ">":
        prmd.append(0)
    else:
        x = x.replace("\n", "")
        prm.append(x)
        
# Converting the array into DataFrames
prm = pd.DataFrame(prm)

# Attaching the "Disease" label column to the input
prm["Disease"] = prmd

# Renaming the columns
prm = prm.rename(index=str, columns={0: "Sequence", "Disease": "Disease"})

# clearing the memory
del prmd

In [ ]:
print("The combined shape of the given data is:", str(prm.shape))
print("The length of the combined data is:", str(len(prm.index)))
print("Does the combined data have any null value? ->", prm.isnull().values.any())

prm = prm.dropna(how = 'any',axis = 0) 

# Shuffling the data and then taking a peek
prm = prm.sample(frac = 1)

In [ ]:
# Function to convert sequence strings into k-mer words, default size = 6 (hexamer words)
def getKmers(sequence, size = 6):
    return [sequence[x:x + size].lower() for x in range(len(sequence) - size + 1)]

prm['words'] = prm.apply(lambda x: getKmers(x['Sequence']), axis = 1)
prm = prm.drop('Sequence', axis=1)

prm_texts = list(prm['words'])
for item in range(len(prm_texts)):
    prm_texts[item] = ' '.join(prm_texts[item])

In [ ]:
# Creating y and printing the shape of it
y = prm.iloc[:, 0].values
print("The shape of y is:", y.shape)

prm["Disease"].value_counts().sort_index().plot.bar()

# clearing the memory
del prm

In [ ]:
# Creating the Bag of Words model using CountVectorizer()
# This is equivalent to k-mer counting
# The n-gram size of 4 was previously determined by testing
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(ngram_range = (4,4))
x = cv.fit_transform(prm_texts)

# clearing the memory
del prm_texts

In [ ]:
# Splitting the human dataset into the training set and test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state = 42, stratify=y)

# Printing the shapes of the train and test matrices
print("The shape of x_train is:", X_train.shape)
print("The shape of y_train is:", y_train.shape)
print("The shape of x_test is:", X_test.shape)
print("The shape of y_test is:", y_test.shape)

# clearing the memory
del x
del y

## Keras

In [ ]:
# Importing the Keras libraries
from keras.models import Sequential
from keras.layers import Dense

# define the keras model
model = Sequential()
model.add(Dense(12, input_dim=731, activation='relu'))
model.add(Dense(15, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# compile the keras model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# fit the keras model on the dataset
model.fit(X_train, y_train, epochs=150, batch_size=10)

In [ ]:
# evaluate the keras model
_, accuracy = model.evaluate(X_test, y_test)
print('Accuracy: %.2f' % (accuracy*100))

# clearing the memory
del accuracy
del model
del X_train
del X_test
del y_test
del y_train